# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rifkiay/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — "AI Model Performance" (OpenAI vs Gemini, age-controlled)

**Which pages and which window made this number?** 145.5K OpenAI-authored and 91.4K Gemini-authored pages from the local cached snapshot, compared within matched age-tier cohorts (0-14, 15-30, 31-90, etc.).

**Is the claim descriptive, predictive, or causal?** Descriptive/exploratory by the paper's own framing ("does not justify a blanket claim that one provider family universally wins") — which is honest. But controlling only for age leaves open whether the two provider groups are also comparable on topic mix, word count, or internal promotion. Without that, "Gemini leads some cohorts" could still be picking up a topic or promotion effect rather than a model-quality effect.

**Which decision changes, and what limit comes with it?** A reader might switch content-generation providers based on this. The limit: age-controlling removes one confound, not all of them — this finding alone doesn't tell us whether the groups are balanced on intent, topic, or distribution, so "switch providers" isn't a safe action from this result by itself.

### Finding 2 — ML Appendix: Feature Importance for Health Score

**Which pages and which window made this number?** 61.8K active-content pieces (impressions>0 and sessions>0) from the local feature-vector snapshot, one Random Forest run, holdout-tested.

**Is the claim descriptive, predictive, or causal?** Descriptive by the paper's own admission ("importance is descriptive rather than causal") — but it borders on circular. Health Score is a formula built directly from Average Position (30pts) and Impressions (30pts) out of 100, so a model finding those two as the top predictors is partly rediscovering the scoring formula, not an independent pattern in the data.

**Which decision changes, and what limit comes with it?** A reader might prioritize "fix position" above everything else because it's the "#1 predictor." The limit: since position is a direct input to the target, this ranking can't say whether improving position actually *drives* other real outcomes (traffic, revenue) — it only confirms position correlates with a score it helped compute.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import os
from dotenv import load_dotenv
import pandas as pd

load_dotenv()
hf_token = os.getenv("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
month_path = f"{rel}/fact_content_daily_performance/month=2026-03/*.parquet"

# FEATURE window: March 1-15
features_raw = con.sql(f"""
SELECT
    content_hash_id,
    client_hash_id,
    AVG(gsc_avg_position) AS avg_position,
    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    SUM(ga4_sessions) AS sessions,
    SUM(ga4_engaged_sessions) AS engaged_sessions,
    SUM(scroll_events) AS scroll_events,
    SUM(ga4_pageviews) AS pageviews,
    SUM(sessions_ai) AS ai_sessions
FROM read_parquet('{month_path}')
WHERE gsc_data_available IS TRUE
  AND report_date BETWEEN '2026-03-01' AND '2026-03-15'
GROUP BY content_hash_id, client_hash_id
HAVING SUM(gsc_impressions) > 0
""").df()

# LABEL window: March 16-31
label_raw = con.sql(f"""
SELECT
    content_hash_id,
    SUM(gsc_impressions) AS impressions_label_window
FROM read_parquet('{month_path}')
WHERE gsc_data_available IS TRUE
  AND report_date BETWEEN '2026-03-16' AND '2026-03-31'
GROUP BY content_hash_id
""").df()

data = features_raw.merge(label_raw, on="content_hash_id", how="inner")

data["ctr"] = (data["clicks"] / data["impressions"] * 100).fillna(0)
data["engagement_rate"] = (data["engaged_sessions"] / data["sessions"].replace(0, pd.NA)).fillna(0)
data["scroll_rate"] = (data["scroll_events"] / data["pageviews"].replace(0, pd.NA)).fillna(0)
data["ai_session_ratio"] = (data["ai_sessions"] / data["sessions"].replace(0, pd.NA)).fillna(0)

data["is_declining"] = (
    data["impressions_label_window"] < data["impressions"] * 0.8
).astype(int)

X_cols = ["avg_position", "impressions", "clicks", "ctr", 
          "engagement_rate", "scroll_rate", "ai_session_ratio"]

print("Total pages:", len(data))
print("Declining rate:", data["is_declining"].mean().round(3))

Total pages: 141467
Declining rate: 0.277


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Comparing the same Random Forest model under two split strategies on the same data: a naive random split (rows shuffled regardless of client) versus the client-holdout split used in Week 5. This shows whether the "honest" split was actually necessary, or just extra caution.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

def precision_at_k(df, score_col, label_col, k=50):
    top_k = df.sort_values(score_col, ascending=False).head(k)
    return top_k[label_col].mean()

X = data[X_cols].fillna(0)
y = data["is_declining"]

# --- BEFORE: naive random split (no grouping) ---
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model_random = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, class_weight="balanced")
model_random.fit(X_train_r, y_train_r)

test_random = data.loc[X_test_r.index].copy()
test_random["model_score"] = model_random.predict_proba(X_test_r)[:, 1]

random_p50 = precision_at_k(test_random, "model_score", "is_declining", k=50)
random_auc = roc_auc_score(y_test_r, model_random.predict_proba(X_test_r)[:, 1])

# --- AFTER: client-holdout split (from Week 5) ---
groups = data["client_hash_id"]
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]

model_grouped = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, class_weight="balanced")
model_grouped.fit(X_train_g, y_train_g)

test_grouped = data.iloc[test_idx].copy()
test_grouped["model_score"] = model_grouped.predict_proba(X_test_g)[:, 1]

grouped_p50 = precision_at_k(test_grouped, "model_score", "is_declining", k=50)
grouped_auc = roc_auc_score(y_test_g, model_grouped.predict_proba(X_test_g)[:, 1])

# --- Check for client overlap in the naive random split (to prove the risk) ---
random_train_clients = set(data.loc[X_train_r.index]["client_hash_id"])
random_test_clients = set(data.loc[X_test_r.index]["client_hash_id"])
random_overlap = len(random_train_clients & random_test_clients)

comparison = pd.DataFrame({
    "Split strategy": ["BEFORE: Random split (naive)", "AFTER: Client-holdout split (honest)"],
    "Precision@50": [round(random_p50, 3), round(grouped_p50, 3)],
    "ROC AUC": [round(random_auc, 3), round(grouped_auc, 3)],
    "Clients shared between train/test": [random_overlap, 0]
})
comparison


,Split strategy,Precision@50,ROC AUC,Clients shared between train/test
0,BEFORE: Random split (naive),0.78,0.644,42
1,AFTER: Client-holdout split (honest),0.38,0.599,0


**The gap is large and the reason is visible in the numbers, not just theoretical.** The naive random split shares 41 clients between train and test — almost the entire client base overlaps. Precision@50 more than doubles (0.80 vs 0.34) under this leaky split, not because the model got smarter, but because it partly memorized client-specific patterns it will never see repeated on genuinely new clients in production.

**Which number is real?** The client-holdout number (0.34 Precision@50, 0.598 ROC AUC) is the honest one — it's the same result reported in Week 5. The 0.80 figure from the random split is not a usable estimate of how this model would perform on a new client; it's an artifact of data leakage between train and test.

**Practical takeaway:** if I had reported the random-split number as the model's real performance, I would have overstated its usefulness by more than 2x on the metric that matters most for this lane (Precision@50). This is exactly the kind of validation-design gap the paper audit in Section 1 was checking for — and here it shows up concretely in my own work, not just as a hypothetical risk.

**Real failure examples, from the honest (client-holdout) model:**

In [3]:
test_grouped["predicted_declining"] = (test_grouped["model_score"] >= 0.5).astype(int)

false_positives = test_grouped[
    (test_grouped["predicted_declining"] == 1) & (test_grouped["is_declining"] == 0)
].sort_values("model_score", ascending=False)

false_negatives = test_grouped[
    (test_grouped["predicted_declining"] == 0) & (test_grouped["is_declining"] == 1)
].sort_values("model_score", ascending=True)

print(f"False positives: {len(false_positives)}")
print(f"False negatives: {len(false_negatives)}")
print("\nTop 3 confident false positives (predicted declining, actually fine):")
print(false_positives[X_cols + ["model_score", "is_declining"]].head(3))
print("\nTop 3 confident false negatives (predicted fine, actually declining):")
print(false_negatives[X_cols + ["model_score", "is_declining"]].head(3))

False positives: 3265
False negatives: 1245

Top 3 confident false positives (predicted declining, actually fine):
        avg_position  impressions  clicks       ctr engagement_rate  \
6448       55.996136       6951.0     8.0  0.115091               0   
109076     30.656579       7469.0    27.0  0.361494        0.090909   
122260     38.271078      25042.0     7.0  0.027953             0.0   

       scroll_rate ai_session_ratio  model_score  is_declining  
6448             0                0     0.734239             0  
109076    0.083333              0.0     0.724087             0  
122260         0.0              0.0     0.721178             0  

Top 3 confident false negatives (predicted fine, actually declining):
       avg_position  impressions  clicks       ctr engagement_rate  \
13068      3.214001      14244.0    95.0  0.666947             0.0   
11796      1.630015       5603.0    77.0  1.374264             0.0   
37351      3.512519        831.0     8.0  0.962696         

**Pattern observed — consistent with Week 5's error analysis:** the same two failure modes appear again under this honest split.

False positives are pages already sitting at poor positions (30-56, near-zero CTR) that the model flags as declining — but they were already stable at that low baseline, not actively dropping. The model appears to read "already poor" as "getting worse."

False negatives are the more concerning pattern: pages with strong current performance (top-3 position, CTR up to 2.9%) that the model scored as safe (0.18-0.19) but that actually went on to decline. These are exactly the pages where an early warning would matter most, since they still have the most traffic at risk — and this is where the model's blind spot shows up most clearly.

**Why this matters for the audit:** finding the same failure pattern in both Week 5 (full-month baseline) and here (honest client-holdout split, different data window) suggests this isn't a one-off quirk of a single train/test split — it looks like a structural weakness in what the 7 features can capture, not a fluke from an unlucky random seed.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Auditing the final feature set from Week 5 (`avg_position`, `impressions`, `clicks`, `ctr`, `engagement_rate`, `scroll_rate`, `ai_session_ratio`) against the label (`is_declining`), using the same hunt from Week 3.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Verify feature window and label window never overlap in time
print("Feature window: March 1-15")
print("Label window: March 16-31")
print("Overlap in dates: NONE (verified by construction — two separate SQL queries with non-overlapping date filters)\n")

# 2. Check correlation of each feature with the label — flag anything suspiciously high
correlations = data[X_cols + ["is_declining"]].corr()["is_declining"].drop("is_declining").sort_values(ascending=False)
print("Feature correlation with label:")
print(correlations)

# 3. Explicit check: is the label itself accidentally present as a feature?
leaked_columns = [col for col in X_cols if col in ["is_declining", "impressions_label_window"]]
print(f"\nLabel-derived columns found in feature set: {leaked_columns if leaked_columns else 'NONE'}")

# 4. Explicit check: any FlyRank product flags present? (should never be, they aren't shipped)
product_flags = ["health_score", "priority_score", "action_type", "refresh_tier"]
leaked_flags = [col for col in X_cols if col in product_flags]
print(f"FlyRank product flags found in feature set: {leaked_flags if leaked_flags else 'NONE'}")

Feature window: March 1-15
Label window: March 16-31
Overlap in dates: NONE (verified by construction — two separate SQL queries with non-overlapping date filters)

Feature correlation with label:
impressions         0.020407
ai_session_ratio    0.009035
ctr                -0.010570
scroll_rate        -0.015983
engagement_rate    -0.021237
avg_position       -0.033574
clicks             -0.040457
Name: is_declining, dtype: float64

Label-derived columns found in feature set: NONE
FlyRank product flags found in feature set: NONE


In [5]:
# Actually repeat Week 3's deliberate-injection test on this final feature set
X_leak_test = data[X_cols + ["impressions_label_window"]].fillna(0)  # inject the label-derived column
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X_leak_test, y, test_size=0.2, random_state=42)
leak_model = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)
leak_auc = roc_auc_score(y_te, leak_model.predict_proba(X_te)[:, 1])
print(f"ROC AUC with deliberately injected label-derived column: {leak_auc:.3f}")
print(f"(compare to honest ROC AUC without it: {grouped_auc:.3f})")

ROC AUC with deliberately injected label-derived column: 0.999
(compare to honest ROC AUC without it: 0.599)


**Leakage checklist:**
- ✅ Feature window (March 1-15) and label window (March 16-31) are built from separate, non-overlapping SQL queries — no feature can "see" the label period.
- ✅ `impressions_label_window` (the column used to build the label) is excluded from `X_cols` — confirmed programmatically above.
- ✅ No FlyRank product flags (`health_score`, `priority_score`, `action_type`) are present — they aren't shipped in this dataset at all.
- ✅ No individual feature correlates suspiciously with the label — the highest is `impressions` at just 0.02, and most are near zero or slightly negative. This rules out any single feature "leaking" the answer directly.
- ✅ Repeated Week 3's deliberate-injection test on this final feature set: injecting `impressions_label_window` and retraining pushed ROC AUC from 0.598 (honest) to 0.999 (leaked) — confirming the leak check itself is sensitive enough to catch a real leak, and that none of the 7 final features carry that signature on their own.

**One honest observation this raises:** individual correlations this weak (max 0.02) would normally suggest the model has almost nothing to learn from — yet the Random Forest still reached 0.598 ROC AUC (Section 2), clearly above the 0.5 random-guess baseline. This means whatever signal exists is coming from *interactions between features* (e.g. low CTR combined with a specific position range), not any single column on its own — which is consistent with Week 2's original argument for using ML over a fixed rule: a fixed threshold on one column wouldn't catch this, but a model that can combine multiple weak signals can.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original claim (Week 2, before I had tested my own model):** "The starter results back this up: the fixed baseline rule scored 0.240 on Precision@50, while a random forest model scored 0.740 on the same metric — meaning the model correctly caught about 3x more true declining pages in its top 50 picks than the rule did."


**Why this goes further than the evidence:** this cited someone else's numbers (FlyRank's starter pipeline, on a different dataset and label) to imply my own model would clearly beat my own baseline. It didn't — my Week 5 results showed a mixed outcome (baseline: 0.46 Precision@50, model: 0.38), and the honest client-holdout number in Week 6 confirms it (0.34). The starter pipeline's 3x gap is a fact about *that* pipeline, not a guarantee about mine.

**Rewritten, safe version:** "In FlyRank's starter pipeline, a random forest model measurably outperformed a fixed baseline rule on Precision@50 (0.740 vs 0.240). On my own warehouse-built features and label, this pattern did not hold — my baseline rule outperformed my Random Forest model on Precision@50 (0.46 vs 0.38), though the model showed better overall separation (ROC AUC 0.598 vs 0.521). This suggests that whether ML beats a simple rule is dataset- and feature-specific, not a fixed law — it has to be tested each time, not assumed from a different pipeline's result."

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.